<a href="https://colab.research.google.com/github/timothyow/research_agent_crew/blob/main/Deep_research_c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automatic Deep Research

Welcome to this new practice lab! By now you should have a clearer view of the elements that compose a multi-agent system. In this lab you will get to put it into action by creating your first crew.

**What you'll learn:**
- How to define agents with specific roles and expertise
- How to provide agents with tools to perform their tasks
- How to create your own tasks that agents will execute
- How to assemble agents and tasks into a Crew, all using CrewAI

## Background

As a research consultant, you're constantly tasked with producing comprehensive reports on diverse topics for demanding clients. You need to build an automatic deep research solution that can rapidly gather, verify, and synthesize information from across the internet, delivering reliable, fact-checked reports that meet tight deadlines and exacting standards regardless of the subject matter.

## General instructions
In this lab you will be presented with a structure of the code, but you will need to complete some of it.

To successfully run this lab, replace all instances of the placeholder `None` with your own code. Sections where you need to write code will be delimited between `### START CODE HERE ###` and `### END CODE HERE ###`.

**<font color='#5DADEC'>Please make sure to save your work periodically, so you don't lose any progress.</font>**

## Table of contents

- [1. Understanding the problem](#1)
- [2. Set up your notebook](#2)
- [3. Define the Agents](#3)
  - [3.1. Create tool instances](#3-1)
  - [3.2. Define the Research Planner agent](#3-2)
  - [3.3. Define the remaining agents](#3-3)
- [4. Create the Tasks](#4)
  - [4.1. Define the Create research plan task](#4-1)
  - [4.2. Define the remaining tasks](#4-2)
- [5. Define the Crew and get the results](#5)

<a id="1"></a>

## 1. Understanding the problem
In this lab, you will focus on building a custom deep research crew. This Crew will be in charge of creating a research plan based on the user's input, and executing it, while reviewing and checking the facts. Finally, with the gathered information a report needs to be generated.

Take some time to decompose the problem into different tasks. Who would be the appropriate "person" to solve each task?

Once you've done your thinking, click below to find an agent/task diagram for this lab.    


<details>    
<summary>
    <font size="3" color="#237b946b"><b>Diagram</b></font>
</summary>

<img src="../images/lab2-agents-tasks-diagram.PNG">

<a id="2"></a>

## 2. Set up your notebook

Before you start coding, run the next two cells to import all necessary modules and configure the environment variables.

In [6]:
!pip install crewai
!pip install crewai_tools
!pip install langchain_openai
!pip install langchain_community
!pip install exa_py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.1/755.1 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.2/39.2 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.9/328.9 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.1/485.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.6/229.6 kB 16.6 MB/s eta 0:00:00


In [10]:
!pip install 'crewai[tools]'

In [7]:
from crewai import Agent, Task, Crew, LLM
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

os.environ["CREWAI_TESTING"] = "true"

# set the OpenAI model (gpt-4o)
os.environ["MODEL"] = "gpt-4o-mini"
# set up the OpenAI API key
os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)

Enter OpenAI API key: ··········


<a id="3"></a>

## 3. Define the Agents

Based on the diagram, you should have four agents:
- **Research Planner**: its goal is to analyze queries and break them down into smaller, specific research topics.
- **Internet Researcher**: its job is to perform research tasks.
- **Fact checker**: its goal is to review information for fact accuracy to avoid misinformation.
- **Report Writer**: is in charge of writing reports, based on gathered information.

<a id="3-1"></a>

### 3.1. Create tool instances
As you can see in the diagram, you will be providing the **Internet Researcher Agent** with tools, so that it can better do their job. In particular, you will give this agent access to search the internet and scrape information from the retrieved webpages.

There are different tools inside CrewAI you can use to search the web, in this lab you will use the [**EXA Search Web Loader**](https://docs.crewai.com/en/tools/search-research/exasearchtool#exa-search-web-loader) tool, which is designed to perform a semantic search for a specified query from a text’s content across the internet. It utilizes the [exa.ai](https://exa.ai/) API to fetch and display the most relevant search results based on the query provided by the user. exa.ai enhances semantic search by capturing richer contextual relationships between concepts, allowing for more precise information retrieval than conventional embedding approaches.

For webscraping, you will use the [**Scrape Website**](https://docs.crewai.com/en/tools/web-scraping/scrapewebsitetool) tool, which is designed to extract and read the content of a specified website.

In the next cell you will define instances of these tools, so you can later assign them to the agents.

In [12]:
# import the tools
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
import os
from getpass import getpass

# set the exa API key
os.environ["EXA_API_KEY"] = getpass("Enter EXA API key: ")

### START CODE HERE ###

# Create the EXASearchTool instance
exa_search_tool = EXASearchTool()
# Create the ScrapeWebsiteTool instance
sccrape_website_tool = ScrapeWebsiteTool()

### END CODE HERE ###

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.3 MB/s eta 0:00:00
Enter EXA API key: ··········


<a id="3-2"></a>

### 3.2. Define the Research Planner agent

In the cell below, you will see how you can create the first agent. This time, all the parameters are set up for you. Here is a quick recap of what each of the parameters represent:

- `Role`: If this was a person doing the job, what title would they have?
- `Goal`: What is the goal this agent in particular is trying to accomplish? Make sure to write concrete goal
- `Background`: it should be something the highlights the skills of the agent relevant to its role. Make sure to use keywords that will actually help your agent get better results.

In [13]:
# define the research planner agent
research_planner = Agent(
    role="Research Planner",
    goal="Analyze queries and break them down into smaller, specific research topics.",
    backstory=(
         "You are a research strategist who excels at breaking down complex questions "
         "into manageable research components. You identify what needs to be researched "
         "and create clear research objectives."
    ),
    verbose=True # set to True to see detailed agent actions
)

<a id="3-3"></a>

### 3.3. Define the remaining agents

Now you can define the three remaining agents. The `role` and `goal` parameters are already filled in for you; use your own creativity to fill in the `backstory`.  

Do not forget to assign the tools to the **Internet Researcher** and **Fact Checker** agents. You can do this by setting the `tools` argument.

In [16]:
researcher = Agent(
    role="Internet Researcher",
    goal="Research thoroughly all assigned topics",
    ### START CODE HERE ###
    backstory=(
        "You are a meticulous and resourceful internet researcher, skilled at digging deep "
        "into various sources to find accurate and comprehensive information."
    ),
    # add the 2 tool instances you created
    tools=[exa_search_tool, sccrape_website_tool],
    ### END CODE HERE ###
    verbose=True
)

fact_checker = Agent(
    role="Fact Checker",
    goal=(
        "Verify data for accuracy, identify inconsistencies, date accuracy, "
        "and flag potential misinformation"
    ),
    ### START CODE HERE ###
    backstory=(
        "You are an expert fact checker, meticulously scrutinizing research findings "
        "for accuracy, bias, and completeness. You excel at identifying misleading information "
        "and ensuring the integrity of all data."
    ),
    tools=[exa_search_tool, sccrape_website_tool],
    ### END CODE HERE ###
    verbose=True
)

report_writer = Agent(
    role="Report Writer",
    goal="Write clear, concise, and well-structured reports based on gathered information",
    ### START CODE HERE ###
    backstory=(
         "You are a master communicator, transforming complex research into elegant, understandable reports. Your expertise lies in structuring information, highlighting key findings, and ensuring every report is a compelling narrative."
    ),
    ### END CODE HERE ###
    verbose=True
)

<a id="4"></a>

## 4. Create the Tasks

Now that you have set up the agents, it is time to define the tasks. If you go back to the diagram, you will see you need four tasks:

- **Create research plan**: Based on the user's query, break it down into specific topics and key questions, and create a focused research plan.
    - Output: A research plan with main research topics to investigate, key questions for each topic, and success criteria for the research.

- **Gather research data**: Using the research plan, collect information on all identified topics. Cite all sources used.
    - Output: Comprehensive research data including: information for each research topic, and citations used along with source credibility notes.

- **Verify information quality**: Review all collected research. Identify any conflicting information, potential misinformation, or gaps that need addressing.
    - Output: A report with the all the collected data, and its review. It should include consistency check results and source reliability ratings

- **Write final report**: Create a comprehensive report that answers the original query using all verified research data. Structure it with clear sections, include citations, and provide actionable insights.
    - Output: The final research report. In addition to the full answer, it should have an executive summary, and complete source citations.


For each `Task` you need to define the following parameters:
- `description`: A thorough description of the task. You can even break it down into different items.
- `expected_output`: what should the output return. Be specific, specially if you want any structure in your result, like a dictionary with specific keys.
- `agent`: who is performing the task? You need to match the task to one of the agents you already defined

In the description you will need to pass the inputs to the tasks. In this lab, you will only have as input the user's query, which will be saved as `user_query`:


<a id="4-1"></a>

### 4.1. Define the Create research plan task

In the cell below, you will see how you can create the first task. This time, all the parameters are set up for you.

In [17]:
# define the create research plan task
create_research_plan_task = Task(
    description=(
        "Based on the user's query, break it down into specific topics and key questions, "
        "and create a focused research plan."
        "The user's query is: {user_query}"
    ),
    expected_output=(
        "A research plan with main research topics to investigate, "
        "key questions for each topic, and success criteria for the research."
        ),
    agent=research_planner,
)

<a id="4-2"></a>

### 4.2. Define the remaining tasks

Now define the three remaining tasks. The `description` is already filled in for you, you will need to define the `expected_output` and `agent` for each of the Tasks.

In [19]:
# define the gather research data task
gather_research_data_task = Task(
    description=(
        "Using the research plan, collect information on all identified topics. "
        "Cite all sources used."
    ),
    ### START CODE HERE ###
    expected_output=(
        "Comprehensive research data including: information for each research topic, and citations used along with source credibility notes."
    ),
    agent=researcher
    ### END CODE HERE ###
)

#define the verify information quality task
verify_information_quality_task = Task(
    description=(
        "Review all collected research. Identify any conflicting information, "
        "potential misinformation, or gaps that need addressing."
    ),
    ### START CODE HERE ###
    expected_output=(
        "A report with all the collected data, and its review. It should include consistency check results and source reliability ratings."
    ),
    agent=fact_checker
    ### END CODE HERE ###
)

# define the write final report task
write_final_report_task = Task(
    description=(
        "Create a comprehensive report that answers the original query using all verified research data. "
        "Structure it with clear sections, include citations, and provide actionable insights."
    ),
    ### START CODE HERE ###
    expected_output=(
        "The final research report. In addition to the full answer, it should have an executive summary, and complete source citations."
    ),
    agent=report_writer
    ### END CODE HERE ###
)

<a id="5"></a>

## 5. Define the Crew and get the results

Once the agents and tasks have been defined, you are ready to create the crew. In order to so, you will need to set the following arguments:
- `agents`: list of agents in the crew
- `tasks`: list of tasks in the crew. The tasks should be listed in the order they should be executed

In the next cell, fill in the agents and tasks for the crew.

In [20]:
# create the crew with the defined agents and tasks
crew = Crew(
    ### START CODE HERE ###
    agents=[research_planner, researcher, fact_checker, report_writer],
    tasks=[create_research_plan_task, gather_research_data_task, verify_information_quality_task, write_final_report_task]
    ###
)

Before running the crew, you need to define the query, which will be used as input for the tasks.

In [21]:
### START CODE HERE ###

# Write your query, which will be used as input for the tasks.
user_query = "What is causing evaporated milk category to decline in Philippines over the past 3 years."

### END CODE HERE ###

Now you are only left with kickstarting the crew to get the results. Since you set `verbose=True` in the agents, you should monitor all the process.

In [22]:
result = crew.kickoff(
    inputs={
        "user_query": user_query,
    }
)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Based on the user's query, break it down into specific topics and key questions, and create a focused    │
│  research plan.The user's query is: What is causing evaporated milk category to decline in Philippines over     │
│  the past 3 years.                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Research Plan: Decline of Evaporated Milk Category in the Philippines**                                      │
│                                                                                                                 │
│  **Main Research Topics to Investigate:**                                                                       │
│                                                                                                                 │
│  1. **Market Trends and Consumer Preferences**                                                                  │
│     - **Key Questions:**                                                                                        │
│       - What are the current market trends in the evaporated milk category over the past three years?           │
│       - How have consumer preferences shifted regarding dairy and non-dairy alternatives?                       │
│       - Are there demographic shifts (age, income level, etc.) influencing purchasing patterns?                 │
│     - **Success Criteria:**                                                                                     │
│       - Identification of at least three major trends affecting evaporated milk sales.                          │
│       - Clear understanding of consumer preference shifts, supported by demographic data.                       │
│                                                                                                                 │
│  2. **Competitive Landscape Analysis**                                                                          │
│     - **Key Questions:**                                                                                        │
│       - What brands are currently leading in the evaporated milk market, and how have their strategies          │
│  changed?                                                                                                       │
│       - Are there new entrants or substitutes that have emerged in the market?                                  │
│       - How do the prices of evaporated milk compare to alternative products?                                   │
│     - **Success Criteria:**                                                                                     │
│       - Detailed competitive analysis that identifies key competitors and their market strategies.              │
│       - Comparative pricing data illustrating the value proposition of evaporated milk versus alternatives.     │
│                                                                                                                 │
│  3. **Regulatory and Economic Factors**                                                                         │
│     - **Key Questions:**                                                                                        │
│       - What regulatory changes have occurred in the dairy sector that may affect evaporated milk production    │
│  and sales?                                                                                                     │
│       - How have economic factors, such as inflation, impacted consumer purchasing power for evaporated milk?   │
│       - Is there a correlation between changes in import/export regulations and product availability?           │
│     - **Success Criteria:**                            

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, collect information on all identified topics. Cite all sources used.            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Thought: I need to gather detailed and accurate information on the decline of the evaporated milk category in  │
│  the Philippines, addressing all identified research topics and their corresponding questions.                  │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "evaporated milk market trends consumer preferences Philippines 2023",                       │
│    "start_published_date": "2020-01-01",                                                                        │
│    "end_published_date": "2023-12-31",                                                                          │
│    "include_domains": null                                                                                      │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Title: Philippines Evaporated Milk Market (2025-2031) Outlook                                                  │
│  URL: https://www.6wresearch.com/industry-report/philippines-evaporated-milk-market-outlook                     │
│  ID: https://www.6wresearch.com/industry-report/philippines-evaporated-milk-market-outlook                      │
│  Score: None                                                                                                    │
│  Published Date: None                                                                                           │
│  Author:                                                                                                        │
│  Image: None                                                                                                    │
│  Favicon: None                                                                                                  │
│  Extras: None                                                                                                   │
│  Subpages: None                                                                                                 │
│  Text:                                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                       

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Checker                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Comprehensive Report on the Decline of Evaporated Milk Category in the Philippines                          │
│                                                                                                                 │
│  ### 1. Market Trends and Consumer Preferences                                                                  │
│  - **Current Market Trends**: The market for evaporated milk is projected to grow from around $3.2 billion in   │
│  2024 to approximately $5.5 billion by 2034, largely due to increasing consumer demand for convenient dairy     │
│  products (Global Insight Services, 2023).                                                                      │
│  - **Consumer Preferences**: There's a noted shift towards non-dairy alternatives, driven by health             │
│  consciousness. This has led to a decline in evaporated milk, historically used in many Filipino households     │
│  (FAS USDA, 2023).                                                                                              │
│  - **Demographics**: Urbanization and a growing middle-class reflect changing dietary habits that demand        │
│  diversified products, including plant-based alternatives (Philippines Dairy and Products Annual, 2023).        │
│                                                                                                                 │
│  ### 2. Competitive Landscape Analysis                                                                          │
│  - **Key Competitors**: Leading brands are Nestlé, Alaska Milk Corporation, and FrieslandCampina. Notably,      │
│  Nestlé dominates the sector due to its marketing prowess and product diversity (Astute Analytica, 2023).       │
│  - **Pricing**: Evaporated milk prices range from $1 to $3 per can, while non-dairy alternatives often cost     │
│  less, heightening competition (Global Insight Services, 2023).                                                 │
│  - **Emergence of Alternatives**: The rise of dairy-free and lactose-free options is critical among             │
│  health-aware younger consumers, challenging traditional evaporated milk appeal (Astute Analytica, 2023).       │
│                                                                                                                 │
│  ### 3. Regulatory and Economic Factors                                                                         │
│  - **Regulatory Landscape**: Government regulations are aimed at enhancing quality standards and managing       │
│  imports. However, around 99% of dairy consumed is still imported due to insufficient local production (FAO,    │
│  2023).                                                                                                         │
│  - **Economic Impact**: Inflation affects purchasing power, prompting some consumers to shift to cheaper,       │
│  plant-based options (FAS USDA, 2023).                                                                          │
│  - **Trade Policies**: Local dairy production faces hurdles from international tariffs and varying              │
│  import/export regulations, influencing availability and pricing (FAO, 2023).                                   │
│                                                                                                                 │
│  ### 4. Marketing and Branding Impacts                 

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Create a comprehensive report that answers the original query using all verified research data.          │
│  Structure it with clear sections, include citations, and provide actionable insights.                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Report on the Decline of the Evaporated Milk Category in the Philippines                       │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│  This report provides an in-depth analysis of the declining market for evaporated milk in the Philippines,      │
│  elucidating the key market trends, consumer preferences, competitive landscape, regulatory and economic        │
│  factors, as well as shifts in health trends and dietary preferences. The findings indicate that while          │
│  traditional evaporated milk has historically played a significant role in Filipino households, emerging        │
│  health consciousness and an increased interest in plant-based alternatives are contributing to its decline.    │
│  The report also highlights actionable insights for stakeholders aimed at revitalizing the evaporated milk      │
│  category.                                                                                                      │
│                                                                                                                 │
│  ## Table of Contents                                                                                           │
│  1. Market Trends and Consumer Preferences                                                                      │
│      - Current Market Trends                                                                                    │
│      - Shift in Consumer Preferences                                                                            │
│      - Demographics Influencing Purchases                                                                       │
│  2. Competitive Landscape Analysis                                                                              │
│      - Key Competitors                                                                                          │
│      - Price Comparisons                                                                                        │
│      - Emergence of Alternatives                                                                                │
│  3. Regulatory and Economic Factors                                                                             │
│      - Regulatory Changes                                                                                       │
│      - Economic Influences on Demand                                                                            │
│      - Impact of Trade Policies                                                                                 │
│  4. Marketing and Branding Impacts                                                                              │
│      - Evolution of Marketing                                                                                   │
│      - Success of Social Media Campaigns                                                                        │
│  5. Health Trends and Dietary Preferences                                                                       │
│      - Current Health Trends                                                                                    │
│      - Changing Dietary Preferences                    



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

From the output of the previous cell check all the outputs for each task. Do they match what you expected? If not, go back and refine the `expected_output`.

You can also print the final report to see the final result of the crew

In [23]:
from IPython.display import Markdown
Markdown(result.raw)

# Comprehensive Report on the Decline of the Evaporated Milk Category in the Philippines

## Executive Summary
This report provides an in-depth analysis of the declining market for evaporated milk in the Philippines, elucidating the key market trends, consumer preferences, competitive landscape, regulatory and economic factors, as well as shifts in health trends and dietary preferences. The findings indicate that while traditional evaporated milk has historically played a significant role in Filipino households, emerging health consciousness and an increased interest in plant-based alternatives are contributing to its decline. The report also highlights actionable insights for stakeholders aimed at revitalizing the evaporated milk category.

## Table of Contents
1. Market Trends and Consumer Preferences
    - Current Market Trends
    - Shift in Consumer Preferences
    - Demographics Influencing Purchases
2. Competitive Landscape Analysis
    - Key Competitors
    - Price Comparisons
    - Emergence of Alternatives
3. Regulatory and Economic Factors
    - Regulatory Changes
    - Economic Influences on Demand
    - Impact of Trade Policies
4. Marketing and Branding Impacts
    - Evolution of Marketing
    - Success of Social Media Campaigns
5. Health Trends and Dietary Preferences
    - Current Health Trends
    - Changing Dietary Preferences
    - Perception of Dairy's Health Benefits
6. Conclusion
7. References

### 1. Market Trends and Consumer Preferences
- **Current Market Trends**: The evaporated milk market is projected to grow from approximately $3.2 billion in 2024 to about $5.5 billion by 2034, driven by increasing consumer demand for convenient dairy products (Global Insight Services, 2023).
  
- **Shift in Consumer Preferences**: Consumers have increasingly shifted towards non-dairy alternatives due to rising health consciousness. This has resulted in a decline in evaporated milk consumption, despite its historical use as a versatile ingredient in many Filipino households (FAS USDA, 2023).

- **Demographics Influencing Purchases**: Continuous urbanization and the growing middle-class population in the Philippines are reflected in changing dietary habits, leading to higher demands for diversified products, which include plant-based alternatives (Philippines Dairy and Products Annual, 2023).

### 2. Competitive Landscape Analysis
- **Key Competitors**: Major brands in the evaporated milk market include Nestlé, Alaska Milk Corporation, and FrieslandCampina, with Nestlé being a leading player owing to its strong marketing and product range (Astute Analytica, 2023).

- **Price Comparisons**: Price points for evaporated milk range from $1 to $3 per can depending on the brand and packaging, while alternatives such as almond milk may continue to be less costly due to lower processing costs (Global Insight Services, 2023).

- **Emergence of Alternatives**: The introduction of dairy-free and lactose-free evaporated milk is gaining traction among health-conscious consumers, making traditional evaporated milk less appealing, particularly among younger demographics (Astute Analytica, 2023).

### 3. Regulatory and Economic Factors
- **Regulatory Changes**: The Filipino dairy sector is influenced by various regulations aimed at improving quality standards and reducing imports, yet local production remains insufficient to meet consumer demand, with around 99% of dairy being imported (FAO, 2023).

- **Economic Influences on Demand**: Inflation has impacted consumer purchasing power, causing some segments to shift towards less expensive, plant-based milk alternatives, thereby affecting the sales of evaporated milk (FAS USDA, 2023).

- **Impact of Trade Policies**: Tariff and trade agreements play a significant role in shaping the market dynamics, with government programs introduced to support local dairy production yet failing to significantly reduce reliance on imports (FAO, 2023).

### 4. Marketing and Branding Impacts
- **Evolution of Marketing**: Companies are adapting their marketing strategies to appeal to health-conscious consumers, focusing on the nutritional benefits of evaporated milk while also addressing dietary shifts towards plant-based options (Fairfield Market Research, 2022).

- **Success of Social Media Campaigns**: Brands leveraging social media have effectively increased awareness and loyalty; however, there is an ongoing challenge to convince a growing portion of consumers to return to traditional dairy products like evaporated milk (Global Insight Services, 2023).

### 5. Health Trends and Dietary Preferences
- **Current Health Trends**: There is significant awareness surrounding health issues related to high sugar and fat content in dairy products. The shift towards low-sugar and fat-free alternatives is evident among younger customers, including the millennial demographic (Brainy Insights, 2021).

- **Changing Dietary Preferences**: Emerging dietary trends, especially vegetarianism and veganism, are creating barriers for the evaporated milk category as consumers opt for non-dairy substitutes instead (Global Insight Services, 2023).

- **Perception of Dairy's Health Benefits**: As the understanding of health benefits evolves, the narrative around the advantages of dairy consumption is being reshaped by the increasing popularity of plant-based alternatives (Astute Analytica, 2023).

### 6. Conclusion
The evaporated milk category in the Philippines is under considerable pressure from changing consumer preferences, economic factors, regulatory landscapes, and evolving health trends. While there are opportunities for growth within the market, stakeholders need to rethink their strategies proactively to adapt to the shifting landscape—particularly by embracing innovations in health-oriented marketing, exploring diversified product lines, and potentially re-engaging with the growing segment of health-conscious consumers.

### 7. References
1. FAS USDA. (2023). Philippines Dairy and Products Annual Report. [Link to source]
2. Global Insight Services. (2023). Evaporated Milk Market Analysis. [Link to source]
3. Astute Analytica. (2023). Market Trends in Dairy Products. [Link to source]
4. FAO. (2023). Dairy Sector Policy Overview. [Link to source]
5. Brainy Insights. (2021). Consumer Health Trend Analysis Report. [Link to source]
6. Fairfield Market Research. (2022). Dairy Marketing Strategies Report. [Link to source]

This comprehensive report aims to provide actionable insights for stakeholders in the evaporated milk industry in the Philippines, addressing the most significant factors contributing to the decline of this product category while offering pathways for potential rejuvenation.